In [2]:
"""
# 03 — Visualización de Feature Maps en CNNs

"Módulo:" 05 — Deep Learning para Clasificación  
"Framework:" TensorFlow / Keras  
"Dataset:" MNIST — modelo entrenado en el notebook 02
---
## ¿Qué vamos a aprender en este notebook?
1. Qué es un feature map y por qué es importante visualizarlo
2. Cómo extraer las activaciones intermedias de una CNN en Keras
3. Visualizar qué detecta cada filtro en cada capa
4. Comparar las representaciones de capas tempranas vs capas profundas
5. Entender por qué las CNNs son tan efectivas para imágenes
---
Este notebook responde una pregunta clave: ¿qué está viendo realmente la red cuando clasifica una imagen?
## 1. ¿Qué es un feature map?

Cuando una imagen pasa por una capa Conv2D, cada filtro produce
una "nueva imagen" llamada feature map (mapa de características).
Input (28×28×1)
↓
Conv2D con 32 filtros
↓
32 feature maps (28×28×32)

Cada feature map responde a una característica distinta de la imagen:
- Filtros en capas tempranas → detectan "bordes, líneas, curvas"
- Filtros en capas profundas → detectan "formas, partes, patrones complejos"

### ¿Por qué visualizarlos?

Las CNNs son frecuentemente tratadas como cajas negras. Visualizar
los feature maps nos permite:

| Beneficio | Descripción |
|---|---|
| "Interpretabilidad" | Entender qué aprendió cada filtro |
| "Debugging" | Detectar filtros muertos o redundantes |
| "Intuición" | Construir criterio para diseñar mejores arquitecturas |
"""

'\n# 03 — Visualización de Feature Maps en CNNs\n\n"Módulo:" 05 — Deep Learning para Clasificación  \n"Framework:" TensorFlow / Keras  \n"Dataset:" MNIST — modelo entrenado en el notebook 02\n---\n## ¿Qué vamos a aprender en este notebook?\n1. Qué es un feature map y por qué es importante visualizarlo\n2. Cómo extraer las activaciones intermedias de una CNN en Keras\n3. Visualizar qué detecta cada filtro en cada capa\n4. Comparar las representaciones de capas tempranas vs capas profundas\n5. Entender por qué las CNNs son tan efectivas para imágenes\n---\nEste notebook responde una pregunta clave: ¿qué está viendo realmente la red cuando clasifica una imagen?\n## 1. ¿Qué es un feature map?\n\nCuando una imagen pasa por una capa Conv2D, cada filtro produce\nuna "nueva imagen" llamada feature map (mapa de características).\nInput (28×28×1)\n↓\nConv2D con 32 filtros\n↓\n32 feature maps (28×28×32)\n\nCada feature map responde a una característica distinta de la imagen:\n- Filtros en capas t

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

TensorFlow version: 2.21.0
Keras version: 3.14.0


In [4]:
"""
## 2. Cargar el modelo entrenado y los datos

En vez de entrenar un modelo nuevo, vamos a reutilizar el modelo
que guardamos en el notebook 02. Esto nos permite:

- Ahorrar tiempo de entrenamiento
- Trabajar con un modelo que ya sabemos que funciona bien
- Demostrar cómo reutilizar modelos guardados en proyectos reales

También vamos a cargar MNIST para tener imágenes de entrada
sobre las cuales visualizar los feature maps.
"""


'\n## 2. Cargar el modelo entrenado y los datos\n\nEn vez de entrenar un modelo nuevo, vamos a reutilizar el modelo\nque guardamos en el notebook 02. Esto nos permite:\n\n- Ahorrar tiempo de entrenamiento\n- Trabajar con un modelo que ya sabemos que funciona bien\n- Demostrar cómo reutilizar modelos guardados en proyectos reales\n\nTambién vamos a cargar MNIST para tener imágenes de entrada\nsobre las cuales visualizar los feature maps.\n'

In [5]:
# Cargar modelo entrenado en notebook 02
model = keras.models.load_model('../models/02_cnn_mnist.keras')
print("Modelo cargado correctamente")
print()
model.summary()

# Cargar MNIST
(_, _), (X_test, y_test) = keras.datasets.mnist.load_data()

# Preprocesar igual que en el notebook 02
X_test = X_test.astype('float32') / 255.0
X_test = X_test[..., np.newaxis]

print(f"\nX_test shape: {X_test.shape}")

Modelo cargado correctamente



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 14, 14, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 7, 7, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 3136)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         401,536 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,264,928 (4.83 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 843,286 (3.22 MB)


X_test shape: (10000, 28, 28, 1)


In [ ]:
"""
"